# Sample library download — direct DataFrame

Pull CDD Vault collections straight into a pandas DataFrame, no intermediate CSV.

> **Before committing:** clear all outputs. This notebook produces SMILES and compound names in `df.head()` cells, which the project's [data-privacy policy](../CLAUDE.md) keeps local. In Jupyter: **Cell → All Output → Clear**, or run `jupyter nbconvert --clear-output --inplace vignettes/Sample_library_download.ipynb`.

Full CLI reference: [docs/documentation.md](../docs/documentation.md).

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('../python'))

import pandas as pd
from get_library import get_df

## Default usage

Three columns by default (`collection, name, smiles`). One row per (molecule, batch).

In [2]:
df = get_df(vault=7108, collections=['AJ', 'AK'])
df.shape, list(df.columns)

resolved_collections=[('AJ', 931034), ('AK', 931035)]
requested_columns=['collection', 'name', 'smiles']

--- collection name=AJ id=931034 ---
  rows=500
  rows=1000
  rows=1500
  rows=2000
  rows=2500
  rows=3000
  rows=3500
  rows=4000
  rows=4500
  rows=5000
  rows=5500
  rows=6000
  rows=6500
  rows=7000
  rows=7500
  collection_rows=7973

--- collection name=AK id=931035 ---
  collection_rows=25

total_rows=7998
multi_batch_molecules=106 (each emitted >1 row — one per batch)


((7998, 3), ['collection', 'name', 'smiles'])

## Custom columns (mixed namespaces)

Any name resolves across the five-step chain: `collection` (special) → molecule top-level → batch top-level → `molecule_fields` → `batch_fields`. See [docs/documentation.md](../docs/documentation.md) for the full chain and the per-collection UDF catalog.

In [3]:
df_wide = get_df(
    vault=7108,
    collections=['AJ', 'AK'],
    columns=[
        'collection', 'name', 'molecule_batch_identifier',
        'smiles', 'molecular_weight', 'log_p',
        'Subseries', 'Px_anywhere',
        'Lib ID', 'Plate ID', 'Px_screened_anywhere',
    ],
)
df_wide.shape, list(df_wide.columns)

resolved_collections=[('AJ', 931034), ('AK', 931035)]
requested_columns=['collection', 'name', 'molecule_batch_identifier', 'smiles', 'molecular_weight', 'log_p', 'Subseries', 'Px_anywhere', 'Lib ID', 'Plate ID', 'Px_screened_anywhere']

--- collection name=AJ id=931034 ---
  rows=500
  rows=1000
  rows=1500
  rows=2000
  rows=2500
  rows=3000
  rows=3500
  rows=4000
  rows=4500
  rows=5000
  rows=5500
  rows=6000
  rows=6500
  rows=7000
  rows=7500
  collection_rows=7973

--- collection name=AK id=931035 ---
  collection_rows=25

total_rows=7998
multi_batch_molecules=106 (each emitted >1 row — one per batch)


((7998, 11),
 ['collection',
  'name',
  'molecule_batch_identifier',
  'smiles',
  'molecular_weight',
  'log_p',
  'Subseries',
  'Px_anywhere',
  'Lib ID',
  'Plate ID',
  'Px_screened_anywhere'])

## Smoke test

`limit=N` caps rows per collection — useful before a full run.

In [4]:
df_smoke = get_df(
    vault=7108,
    collections=['AJ'],
    columns=['collection', 'name', 'smiles', 'Subseries', 'Lib ID'],
    limit=3,
)
df_smoke.shape, list(df_smoke.columns)

resolved_collections=[('AJ', 931034)]
requested_columns=['collection', 'name', 'smiles', 'Subseries', 'Lib ID']

--- collection name=AJ id=931034 ---
  collection_rows=3

total_rows=3
note: limit set → unmatched-column WARN skipped (false positives likely). Re-run without limit to validate columns, or use --list-fields.


((3, 5), ['collection', 'name', 'smiles', 'Subseries', 'Lib ID'])

## Discovering available fields

To list every column per namespace with coverage counts, use the CLI:

In [ ]:
!python3 ../python/get_library.py --vault 7108 --collections AK --list-fields

## Saving to CSV (optional)

If you also want a CSV on disk, either run the CLI (`python3 ../python/get_library.py ...`) or save from the DataFrame:

In [ ]:
# df_wide.to_csv('../output/library.csv', index=False)